[Reference](https://levelup.gitconnected.com/building-a-high-performance-parallel-llm-pipeline-using-weight-optimization-kv-cache-sdpa-and-d02225f2b1d1)

# Setting up the Environment

In [1]:
# Import necessary libraries from Hugging Face Transformers for model and tokenizer handling
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# Import PyTorch for tensor operations
import torch
# Import time for measuring performance
import time
# Import tqdm for progress bars in Jupyter notebooks
from tqdm.notebook import tqdm
# Import json for handling JSON data
import json

LLM_API_KEY = "YOUR_LLM_API_KEY"  # Replace with your actual API key (OpenAI, HuggingFace, Nebius, Together AI etc.)


# Defining the model ID (We are using Llama-3.2 1B model)
model_id = "meta-llama/Llama-3.2-1B"

In [2]:
import os, re
from openai import OpenAI

# Initialize Nebius OpenAI client
def setup_client(api_key=None):
    key = api_key or os.getenv("NEBIUS_API_KEY")
    return OpenAI(base_url="https://api.studio.nebius.com/v1/", api_key=key) if key else None

# Extract score from text using regex
def extract_score(text):
    for p in [r"(\d+\.?\d*)/1\.0", r"Score:\s*(\d+\.?\d*)", r"(\d+\.?\d*)"]:
        m = re.search(p, text)
        if m:
            s = float(m.group(1))
            return s if s <= 1 else s / 10 if s <= 10 else s / 100
    return 0.0

# Get similarity score between two answers
def get_score(client, gen, truth, model="deepseek-ai/DeepSeek-V3"):
    prompt = "You are a lenient evaluator. Score from 0.0 to 1.0. Only return the number."
    msg = f"[Generated Answer]:\n{gen}\n\n[Ground Truth Answer]:\n{truth}"
    res = client.chat.completions.create(model=model, messages=[
        {"role": "system", "content": prompt},
        {"role": "user", "content": msg}
    ])
    return extract_score(res.choices[0].message.content.strip())

In [3]:
def get_model_memory_footprint(model):
    """
    Get the memory footprint of a model in megabytes.

    Args:
        model: The model to measure memory footprint for.

    Returns:
        float: Memory footprint in megabytes, rounded to 2 decimal places.
    """
    # Get memory footprint in bytes and convert to megabytes
    memory_bytes = model.get_memory_footprint()
    memory_mb = memory_bytes / (1024 * 1024)

    return round(memory_mb, 2)

In [4]:
# Function to generate text from a model and tokenizer with optional memory and time measurement
def generate_text(tokenizer, model, **kwargs):
    """
    Generate text using a given tokenizer and model.

    Args:
        tokenizer: The tokenizer for encoding input text.
        model: The language model for text generation.
        **kwargs: Additional arguments for model.generate(), including:
            - input_text (str): The prompt to generate from.
            - max_new_tokens, do_sample, temperature, top_p, top_k, pad_token_id, etc.

    Returns:
        tuple: A tuple containing the generated text (str), peak memory usage in MB (float or None),
               and generation time in seconds (float).
    """
    # Extract the input text from keyword arguments
    input_text = kwargs.pop('input_text')
    # Tokenize the input text and convert it to PyTorch tensors
    inputs = tokenizer(input_text, return_tensors="pt")

    # Check if a CUDA-enabled GPU is available
    if torch.cuda.is_available():
        # Reset peak memory statistics for the current CUDA device
        torch.cuda.reset_peak_memory_stats()
        # Get the device of the model (e.g., 'cuda:0')
        device = next(model.parameters()).device
        # Move the input tensors to the same device as the model
        inputs = {k: v.to(device) for k, v in inputs.items()}

    # Record the start time for generation
    start_time = time.time()
    # Generate text using the model with the provided inputs and generation arguments
    output_tokens = model.generate(**inputs, **kwargs)
    # Record the end time for generation
    end_time = time.time()

    # Calculate the total generation time and round it to 3 decimal places
    generation_time = round(end_time - start_time, 3)

    # Check if a CUDA-enabled GPU is available to measure memory
    if torch.cuda.is_available():
        # Measure the peak memory allocated on the GPU during generation
        # Convert bytes to megabytes (MB) and round to 3 decimal places
        peak_memory = round(torch.cuda.max_memory_allocated() / (1024 ** 2), 3)
    else:
        # If no GPU is available, set peak memory to None
        peak_memory = None

    # Decode the generated tokens back into a string, skipping special tokens
    generated_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    # Return the generated text, peak memory usage, and generation time
    return generated_text, peak_memory, generation_timea

# Our Evaluation Datasets

In [5]:
# Import the json library to work with JSON files
import json

# --- Load Evaluation Dataset ---
# Open the evaluation data file in read mode
with open('eval_data.json', 'r') as file:
    # Load the JSON content from the file into the 'eval_data' variable
    eval_data = json.load(file)

# --- Display Sample Questions and Answers ---
# Print the first two questions and their answers from the dataset to verify it's loaded correctly
print("Sample Question 1:", eval_data[0]['q'])
print("Sample Answer 1:", eval_data[0]['a'])
print("\nSample Question 2:", eval_data[1]['q'])
print("Sample Answer 2:", eval_data[1]['a'])

# --- Total Number of Questions ---
# Print the total number of questions in the evaluation dataset
print("Total Number of Questions:", len(eval_data))

# Evaluating Full Precision Model (Baseline)

In [6]:
# Load the tokenizer for the specified model
model_id = "meta-llama/Llama-3.2-1B"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load the causal language model with bfloat16 precision on CPU
original_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto" # Use "auto" for automatic device mapping if you have a GPU
)

In [7]:
# Running the evaluation on the model with the evaluation dataset
base_model_results = evaluate_model(original_model, tokenizer, eval_data)

# --- Evaluate with LLM Judge ---
# Use the 'evaluate_with_judge' function to assess the model's performance.
# This function takes the model's results and an API key for the judging service.
# It returns a DataFrame with detsailed results and a dictionary of overall metrics.
# Note: You need to replace "your_nebius_api_key_here" with your actual Nebius API key.
base_model_results_df, base_model_metrics = evaluate_with_judge(
    base_model_results,  # The results from your model evaluation
    api_key=LLM_API_KEY,  # API key for the judging service
    model_name="meta-llama/Llama-3.3-70B-Instruct"  # Name of the model being evaluated
)

In [8]:
# --- Display Base Model Metrics ---
# Print the overall performance metrics for the base model.
# These metrics include average latency, memory usage, and similarity score.
print("Base Model Performance Metrics:")
for key, value in base_model_metrics.items():
    # Print each metric with its corresponding value, formatted to 4 decimal places
    print(f"- {key.replace('_', ' ').title()}: {value:.4f}")

# Performing W4A16 Quantization

In [9]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Define quantization config: 4-bit weights (W4A16)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,          # optional: improves accuracy
    bnb_4bit_quant_type="nf4",               # "nf4" (normal float 4-bit) or "fp4"
    bnb_4bit_compute_dtype=torch.bfloat16    # use bfloat16 for activations (W4A16)
)

In [10]:
# Load the model with 4-bit quantization (weights only)
w4a16_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"  # use "auto" for best device placement (GPU/CPU)
)

In [11]:
# Memory footprint of the quantized model
w4a16_model_memory = get_model_memory_footprint(w4a16_model)
print("Quantized Model Memory Footprint (MB):", w4a16_model_memory)

In [12]:
# --- Evaluate the W4A16 Quantized Model ---
# Running the evaluation on the quantized model with the evaluation dataset
w4a16_model_results = evaluate_model(w4a16_model, tokenizer, eval_data)

# --- Evaluate with LLM Judge ---
# Use the 'evaluate_with_judge' function to assess the quantized model's performance.
w4a16_model_results_df, w4a16_model_metrics = evaluate_with_judge(
    w4a16_model_results,  # The results from your quantized model evaluation
    api_key=LLM_API_KEY,  # API key for the judging service
    model_name="meta-llama/Llama-3.3-70B-Instruct"  # Name of the model being evaluated
)

In [13]:
# --- Display W4A16 Quantized Model Metrics ---
# These metrics include average latency, memory usage, and similarity score.
print("W4A16 Quantized Model Performance Metrics:")
for key, value in w4a16_model_metrics.items():
    # Print each metric with its corresponding value, formatted to 4 decimal places
    print(f"- {key.replace('_', ' ').title()}: {value:.4f}")

# Comparing Base vs W4A16 vs W8A8

In [14]:
# Configure for 8-bit weight + activation quantization (W8A8)
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,                           # W8
    llm_int8_threshold=6.0,                      # Optional: threshold for outlier detection
    llm_int8_has_fp16_weight=False,              # Force 8-bit only mode (no fallback to fp16)
    llm_int8_enable_fp32_cpu_offload=True,       # Optional: offload to CPU for better memory management
)

# defining the tokenizer again for clarity
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load the model with 8-bit weights and 8-bit activations
w8a8_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"  # Will use GPU if available
)

In [15]:
# Memory footprint of the quantized model
w8a8_model_memory = get_model_memory_footprint(w8a8_model)
print("Quantized Model Memory Footprint (MB):", w8a8_model_memory)

In [16]:
# --- Evaluate the W8a8 Quantized Model ---
# Running the evaluation on the quantized model with the evaluation dataset
w8a8_model_results = evaluate_model(w8a8_model, tokenizer, eval_data)


# --- Evaluate with LLM Judge ---
# Use the 'evaluate_with_judge' function to assess the quantized model's performance.
w8a8_model_results_df, w4a16_model_metrics = evaluate_with_judge(
    w8a8_model_results,  # The results from your quantized model evaluation
    api_key=LLM_API_KEY,  # API key for the judging service
    model_name="meta-llama/Llama-3.3-70B-Instruct"  # Name of the model being evaluated
)


In [17]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,               # Load weights in 4-bit (W4)
    bnb_4bit_use_double_quant=True, # Optional but recommended for better accuracy
    bnb_4bit_quant_type="nf4",      # NormalFloat4 quantization, good balance between speed and accuracy
    llm_int8_threshold=6.0,          # Optional threshold (can be adjusted or removed)
    llm_int8_has_fp16_weight=False, # Force 4-bit only mode
    llm_int8_enable_fp32_cpu_offload=True, # Optional CPU offload for memory management
)

# # Load the model with 4-bit weights and 8-bit activations
w4a8_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

#
w4a8_model_memory = get_model_memory_footprint(w4a8_model)
print("W4A8 Quantized Model Memory Footprint (MB):", w4a8_model_memory)

In [18]:
# Evaluate W4A8 model results with the judge
w4a8_model_results_df, w4a8_model_metrics = evaluate_with_judge(
    w4a8_model_results,  # results from your W4A8 quantized model evaluation
    api_key=LLM_API_KEY,
    model_name="meta-llama/Llama-3.3-70B-Instruct"
)

# --- Evaluate with LLM Judge ---
# Use the 'evaluate_with_judge' function to assess the quantized model's performance.
w4a8_model_results_df, w4a16_model_metrics = evaluate_with_judge(
    w4a8_model_results_df,  # The results from your quantized model evaluation
    api_key=LLM_API_KEY,  # API key for the judging service
    model_name="meta-llama/Llama-3.3-70B-Instruct"  # Name of the model being evaluated
)